In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn==1.9.1', 'xgboost==3.4.1'], capture_output=True, text=True)
print('PIP', r.returncode, r.stderr[-300:])

In [ ]:
import glob, os, platform
cands = glob.glob('/kaggle/input/**/splits.json', recursive=True)
DATA_DIR = os.path.dirname(cands[0])
import sklearn, xgboost, numpy, scipy
print('DATA', DATA_DIR, 'CPUS', os.cpu_count(), platform.python_version(), 'sklearn', sklearn.__version__, 'xgb', xgboost.__version__, 'numpy', numpy.__version__)

In [ ]:
"""Shared data loading, feature preprocessing and split generation."""
import numpy as np
import pandas as pd

DATA = DATA_DIR
PROC_LEVELS = ['CAST', 'ANNEAL', 'WROUGHT', 'POWDER', 'OTHER', 'UNKNOWN']
DROP_PHYS = ['PHYS:Weight Fraction', 'PHYS:Atomic Fraction', 'PHYS:Total weight']


def load(dataset, data_dir=DATA):
    """Return records dataframe (meta) and full feature frame aligned to records."""
    fname = {'HV': 'hv_records.csv', 'YS': 'ys_records.csv', 'GE': 'ext_ge_records.csv'}[dataset]
    rec = pd.read_csv(f"{data_dir}/{fname}")
    F = pd.read_csv(f"{data_dir}/composition_features.csv")
    F = F.drop(columns=[c for c in DROP_PHYS + ['PHYS_FAIL'] if c in F.columns])
    X = rec[['comp_key']].merge(F, on='comp_key', how='left').drop(columns=['comp_key'])
    assert len(X) == len(rec) and X.notna().all().all(), "feature merge failed"
    for p in PROC_LEVELS:
        X[f"PROC:{p}"] = (rec.proc == p).astype(float).values
    if dataset == 'YS':
        X['TEST:T_C'] = rec.test_T.values
        X['TEST:tension'] = (rec.test == 'T').astype(float).values
    for ph in ['FCC', 'BCC', 'other']:
        X[f"PHASE:{ph}"] = (rec.phase3 == ph).astype(float).values
    X.index = rec.rid.values
    return rec, X


def feature_columns(X, blocks):
    """blocks: iterable of prefixes, e.g. ('FRAC', 'PHYS', 'MAG', 'PROC', 'TEST')."""
    return [c for c in X.columns if c.split(':')[0] in blocks]


DEFAULT_BLOCKS = ('FRAC', 'PHYS', 'MAG', 'PROC', 'TEST')


class Prep:
    """Train-fold-only preprocessing: drop constant columns, drop near-duplicate columns
    (|Pearson r| > 0.98, keep first), standardize. Fitted on training rows only."""

    def __init__(self, corr_thresh=0.98, scale=True, clip=True):
        self.corr_thresh = corr_thresh
        self.scale = scale
        self.clip = clip  # clamp features to the training-fold range (guards linear/NN extrapolation)

    def fit(self, X):
        X = np.asarray(X, float)
        sd = X.std(0)
        keep = np.where(sd > 1e-10)[0]
        Xk = X[:, keep]
        C = np.abs(np.corrcoef(Xk, rowvar=False))
        np.fill_diagonal(C, 0)
        drop = set()
        for i in range(C.shape[0]):
            if i in drop:
                continue
            for j in np.where(C[i, i + 1:] > self.corr_thresh)[0] + i + 1:
                drop.add(j)
        self.keep = keep[[i for i in range(len(keep)) if i not in drop]]
        self.mu = X[:, self.keep].mean(0)
        self.sd = X[:, self.keep].std(0)
        self.lo = X[:, self.keep].min(0)
        self.hi = X[:, self.keep].max(0)
        return self

    def transform(self, X):
        X = np.asarray(X, float)[:, self.keep]
        if self.clip:
            X = np.clip(X, self.lo, self.hi)
        return (X - self.mu) / self.sd if self.scale else X


def balanced_group_folds(groups, n_splits, rng):
    """Random assignment of groups to folds, balancing record counts (largest groups first,
    random tie-breaking). Returns fold id per record."""
    groups = np.asarray(groups)
    ug, counts = np.unique(groups, return_counts=True)
    order = rng.permutation(len(ug))
    order = order[np.argsort(-counts[order], kind='stable')]
    load_ = np.zeros(n_splits)
    gfold = {}
    for gi in order:
        f = int(np.argmin(load_ + rng.random(n_splits) * 1e-6))
        gfold[ug[gi]] = f
        load_[f] += counts[gi]
    return np.array([gfold[g] for g in groups])


def make_splits(rec, dataset, n_rep=3, n_splits=5, seed=2026, min_elem=40, max_frac=0.6):
    """Outer evaluation protocols. Returns list of dicts:
    {protocol, rep, fold, label, train_idx, test_idx}."""
    n = len(rec)
    out = []
    for r in range(n_rep):
        rng = np.random.default_rng(seed + r)
        perm_fold = np.empty(n, int)
        perm_fold[rng.permutation(n)] = np.arange(n) % n_splits
        for proto, fold_id in [('random', perm_fold),
                               ('composition', balanced_group_folds(rec.comp_key.values, n_splits, rng)),
                               ('system', balanced_group_folds(rec.sys_key.values, n_splits, rng))]:
            for f in range(n_splits):
                te = np.where(fold_id == f)[0]
                tr = np.where(fold_id != f)[0]
                out.append(dict(protocol=proto, rep=r, fold=f, label=f"{proto}-r{r}f{f}", train_idx=tr, test_idx=te))
    # leave-one-element-out
    els = pd.Series([e for s in rec.sys_key for e in s.split('-')]).value_counts()
    for el, cnt in els.items():
        frac = cnt / n
        if cnt >= min_elem and frac <= max_frac:
            has = rec.sys_key.apply(lambda s: el in s.split('-')).values
            out.append(dict(protocol='element', rep=0, fold=0, label=f"element-{el}",
                            train_idx=np.where(~has)[0], test_idx=np.where(has)[0]))
    # temporal: train on literature up to 2018, test on 2019 papers
    yr = rec.year.values
    out.append(dict(protocol='temporal', rep=0, fold=0, label='temporal-2019',
                    train_idx=np.where(yr <= 2018)[0], test_idx=np.where(yr == 2019)[0]))
    return out


def inner_groups(rec_tr, protocol):
    """Grouping used for inner (tuning / calibration) folds; mirrors the deployment scenario."""
    if protocol == 'random':
        return None
    if protocol == 'composition':
        return rec_tr.comp_key.values
    return rec_tr.sys_key.values


def metrics(y, p):
    from scipy.stats import spearmanr
    y = np.asarray(y, float); p = np.asarray(p, float)
    e = p - y
    ss = ((y - y.mean()) ** 2).sum()
    return dict(n=len(y), MAE=np.abs(e).mean(), RMSE=np.sqrt((e ** 2).mean()),
                R2=1 - (e ** 2).sum() / ss if ss > 0 else np.nan,
                rho=spearmanr(y, p).correlation if len(y) > 2 else np.nan, bias=e.mean())


In [ ]:
"""Model zoo, nested hyper-parameter tuning, CV+ conformal intervals and applicability domain."""
import itertools
import warnings
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
N_JOBS = 4


class Scaled:
    """Wraps an estimator with train-fold preprocessing and optional target standardization."""

    def __init__(self, est, scale_X=True, scale_y=False):
        self.est, self.scale_X, self.scale_y = est, scale_X, scale_y

    def fit(self, X, y):
        self.prep = Prep(scale=self.scale_X).fit(X)
        y = np.asarray(y, float)
        self.ym, self.ys = (y.mean(), y.std()) if self.scale_y else (0.0, 1.0)
        self.est.fit(self.prep.transform(X), (y - self.ym) / self.ys)
        return self

    def predict(self, X, return_std=False):
        Z = self.prep.transform(X)
        if return_std:
            m, s = self.est.predict(Z, return_std=True)
            return m * self.ys + self.ym, s * self.ys
        return self.est.predict(Z) * self.ys + self.ym


def grid(d):
    keys = list(d)
    return [dict(zip(keys, v)) for v in itertools.product(*[d[k] for k in keys])]


SPACES = {
    'Mean': [dict()],
    'kNN': grid(dict(n_neighbors=[1, 3, 5, 10, 15])),
    'Ridge': [dict()],
    'SVR': grid(dict(C=[1, 10, 100], gamma=[0.002, 0.01, 0.05], epsilon=[0.05, 0.2])),
    'GPR': [dict()],
    'RF': grid(dict(max_features=[0.2, 0.4, 0.7], min_samples_leaf=[1, 3])),
    'XGB': grid(dict(max_depth=[3, 5, 7], learning_rate=[0.03, 0.08], colsample_bytree=[0.3, 0.7],
                     min_child_weight=[1, 4])),
    'MLP': grid(dict(hidden_layer_sizes=[(128,), (256, 128)], alpha=[1e-3, 1e-1, 1.0])),
}
N_ITER = {'SVR': 10, 'RF': 6, 'XGB': 10, 'MLP': 6, 'kNN': 5}


def build(name, p, seed=0):
    if name == 'Mean':
        return Scaled(DummyRegressor(), scale_X=False)
    if name == 'kNN':
        return Scaled(KNeighborsRegressor(weights='distance', **p))
    if name == 'Ridge':
        return Scaled(RidgeCV(alphas=np.logspace(-3, 3, 13)), scale_y=True)
    if name == 'SVR':
        return Scaled(SVR(kernel='rbf', **p), scale_y=True)
    if name == 'GPR':
        k = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(length_scale=10.0, length_scale_bounds=(1e-1, 1e3), nu=2.5) \
            + WhiteKernel(0.1, (1e-4, 1.0))
        return Scaled(GaussianProcessRegressor(kernel=k, normalize_y=False, n_restarts_optimizer=1,
                                               random_state=seed), scale_y=True)
    if name == 'RF':
        return Scaled(RandomForestRegressor(n_estimators=300, n_jobs=N_JOBS, random_state=seed, **p), scale_X=False)
    if name == 'XGB':
        return Scaled(XGBRegressor(n_estimators=500, subsample=0.8, reg_lambda=1.0, tree_method='hist',
                                   n_jobs=N_JOBS, random_state=seed, verbosity=0, **p), scale_X=False)
    if name == 'MLP':
        return Scaled(MLPRegressor(learning_rate_init=1e-3, max_iter=1500, early_stopping=True,
                                   validation_fraction=0.15, n_iter_no_change=30, random_state=seed, **p),
                      scale_y=True)
    raise ValueError(name)


def inner_folds(n, groups, k, seed):
    rng = np.random.default_rng(seed)
    if groups is None:
        f = np.empty(n, int); f[rng.permutation(n)] = np.arange(n) % k
        return f
    return balanced_group_folds(groups, k, rng)


def tune(name, X, y, groups, seed, k=3):
    space = SPACES[name]
    if len(space) == 1:
        return space[0], np.nan
    rng = np.random.default_rng(seed)
    n_iter = min(N_ITER.get(name, len(space)), len(space))
    cand = [space[i] for i in rng.choice(len(space), n_iter, replace=False)]
    fid = inner_folds(len(y), groups, k, seed + 7)
    best, best_s = None, np.inf
    for p in cand:
        se = 0.0
        for f in range(k):
            tr, va = fid != f, fid == f
            m = build(name, p, seed).fit(X[tr], y[tr])
            se += ((m.predict(X[va]) - y[va]) ** 2).sum()
        s = np.sqrt(se / len(y))
        if s < best_s:
            best, best_s = p, s
    return best, best_s


def knn_distance(Ztr, Zte, k=5, exclude_self=False):
    nn = NearestNeighbors(n_neighbors=k + int(exclude_self)).fit(Ztr)
    d, _ = nn.kneighbors(Zte)
    if exclude_self:
        d = d[:, 1:]
    return d.mean(1)


def cv_plus(name, p, X, y, Xte, groups, seed, alphas=(0.1, 0.32), k=5, Zall=None):
    """CV+ (Barber et al., 2021) prediction intervals plus a normalized (difficulty-scaled)
    cross-conformal variant. groups=None -> random inner folds; otherwise grouped inner folds.
    Zall: (Ztr, Zte) standardized descriptor matrices used for the kNN difficulty score."""
    fid = inner_folds(len(y), groups, k, seed + 11)
    mu_te = np.zeros((len(y), len(Xte)))  # mu_{-k(i)}(x_test) for each calibration point i
    R = np.zeros(len(y))
    s_cal = np.zeros(len(y))
    Ztr, Zte = Zall
    for f in range(k):
        tr, va = fid != f, fid == f
        m = build(name, p, seed).fit(X[tr], y[tr])
        R[va] = np.abs(m.predict(X[va]) - y[va])
        mu_te[va] = m.predict(Xte)[None, :]
        s_cal[va] = knn_distance(Ztr[tr], Ztr[va])
    out = {}
    n = len(y)
    for a in alphas:
        lo_k = int(np.floor(a * (n + 1)))
        hi_k = int(np.ceil((1 - a) * (n + 1)))
        lo_k, hi_k = max(lo_k, 1), min(hi_k, n)
        lo = np.sort(mu_te - R[:, None], axis=0)[lo_k - 1]
        hi = np.sort(mu_te + R[:, None], axis=0)[hi_k - 1]
        out[a] = (lo, hi)
    # normalized cross-conformal: scores R_i / s_i ; s(x) for test relative to the full training set
    beta = np.median(s_cal)
    sc = R / (s_cal + beta)
    s_te = knn_distance(Ztr, Zte) + beta
    outn = {}
    for a in alphas:
        q = np.sort(sc)[min(int(np.ceil((1 - a) * (n + 1))), n) - 1]
        outn[a] = (q * s_te,)
    return out, outn


In [ ]:
"""Runner for Kaggle CPU sessions (executed inside the notebook after common.py and models.py cells).
Runs `MODELS` on yield strength over the frozen splits with the same tuning/CV+ code as bench.py and prints
compact per-fold metrics (M|), within-system Spearman (W|) and top-quintile recall on the temporal split (K|)."""
import json, time
import numpy as np, pandas as pd
from scipy.stats import spearmanr


def run_kaggle(MODELS, UQ, DATA, ds='YS'):
    rec, Xdf = load(ds, DATA)
    cols = feature_columns(Xdf, DEFAULT_BLOCKS)
    X = Xdf[cols].values; y = rec.target.values.astype(float)
    rid2i = {r: i for i, r in enumerate(rec.rid.values)}
    splits = json.load(open(f"{DATA}/splits.json"))[ds]
    comp_cols = feature_columns(Xdf, ('FRAC', 'PHYS', 'MAG'))
    Xc = Xdf[comp_cols].values
    allrows = []
    for sp in splits:
        te = np.array([rid2i[r] for r in sp['test']]); mask = np.ones(len(y), bool); mask[te] = False; tr = np.where(mask)[0]
        rtr = rec.iloc[tr]; grp = inner_groups(rtr, sp['protocol'])
        seed = (sum(map(ord, sp['label'])) * 7919) % (2 ** 31)
        pc = Prep(clip=False).fit(Xc[tr]); Ztr, Zte = pc.transform(Xc[tr]), pc.transform(Xc[te])
        for name in MODELS:
            t0 = time.time()
            p, _ = tune(name, X[tr], y[tr], grp, seed)
            m = build(name, p, seed).fit(X[tr], y[tr])
            out = pd.DataFrame({'label': sp['label'], 'protocol': sp['protocol'], 'rep': sp['rep'], 'rid': rec.rid.values[te],
                                'model': name, 'y': y[te]})
            if name == 'GPR':
                out['yhat'], out['gp_sd'] = m.predict(X[te], return_std=True)
            else:
                out['yhat'] = m.predict(X[te])
            if name in UQ:
                variants = [('naive', None)] + ([('matched', grp)] if sp['protocol'] != 'random' else [])
                for vn, g in variants:
                    iv, _ = cv_plus(name, p, X[tr], y[tr], X[te], g, seed, alphas=(0.1, 0.32), Zall=(Ztr, Zte))
                    for a in (0.1, 0.32):
                        tag = int(round((1 - a) * 100)); out[f'cvp_{vn}_lo{tag}'], out[f'cvp_{vn}_hi{tag}'] = iv[a]
            allrows.append(out)
            e = out.yhat - out.y
            print(f"{sp['label']:<22} {name:<5} RMSE={np.sqrt((e**2).mean()):7.1f} t={time.time()-t0:5.1f}s params={p}", flush=True)
    D = pd.concat(allrows, ignore_index=True)
    D.to_csv(f"/kaggle/working/{ds}_{'_'.join(MODELS)}_pred.csv", index=False)
    cov = lambda d, lo, hi: float(((d.y >= d[lo]) & (d.y <= d[hi])).mean())
    lines = ["M|model|label|protocol|n|MAE|RMSE|R2|rho|bias|cov_gp90|w_gp90|cov_gp68|w_gp68|cov_cvn90|w_cvn90|cov_cvn68|w_cvn68|cov_cvm90|w_cvm90|cov_cvm68|w_cvm68"]
    for (model, lab), d in D.groupby(['model', 'label'], sort=False):
        e = d.yhat - d.y; ss = ((d.y - d.y.mean()) ** 2).sum()
        v = [len(d), e.abs().mean(), np.sqrt((e ** 2).mean()), 1 - (e ** 2).sum() / ss, spearmanr(d.y, d.yhat).correlation, e.mean()]
        if model == 'GPR':
            for t, z in ((90, 1.6449), (68, 0.9945)):
                lo, hi = d.yhat - z * d.gp_sd, d.yhat + z * d.gp_sd
                v += [float(((d.y >= lo) & (d.y <= hi)).mean()), float((hi - lo).mean())]
        else:
            v += [np.nan] * 4
        for vn in ('naive', 'matched'):
            for t in (90, 68):
                lo, hi = f'cvp_{vn}_lo{t}', f'cvp_{vn}_hi{t}'
                if lo in d and d[lo].notna().all():
                    v += [cov(d, lo, hi), float((d[hi] - d[lo]).mean())]
                else:
                    v += [np.nan, np.nan]
        lines.append("M|" + "|".join([model, lab, d.protocol.iloc[0]] + [f"{x:.4f}" if isinstance(x, float) else str(x) for x in v]))
    # within-system Spearman (system protocol; room-temperature compression records only)
    cond = rec.set_index('rid')
    S = D[D.protocol == 'system'].copy(); S['sys'] = S.rid.map(cond.sys_key)
    S = S[(S.rid.map(cond.test_T) == 25) & (S.rid.map(cond.test) == 'C')]
    for (model, rep, sk), s in S.groupby(['model', 'rep', 'sys']):
        if len(s) >= 5 and s.y.std() > 0:
            lines.append(f"W|{model}|{rep}|{sk}|{len(s)}|{spearmanr(s.y, s.yhat).correlation:.4f}")
    # top-quintile recall on the temporal split
    for model, g in D[D.label == 'temporal-2019'].groupby('model'):
        k = max(1, int(round(0.2 * len(g)))); top = set(g.nlargest(k, 'y').rid)
        scores = {'mean': g.yhat}
        if 'cvp_matched_lo68' in g and g.cvp_matched_lo68.notna().all():
            scores['UCB'] = g.cvp_matched_hi68; scores['LCB'] = g.cvp_matched_lo68
        for sn, s in scores.items():
            sel = set(g.assign(s=s.values).nlargest(k, 's').rid)
            lines.append(f"K|{model}|temporal-2019|{sn}|{k}|{len(g)}|{len(sel & top) / k:.4f}")
    print('=====RES_BEGIN=====')
    print("\n".join(lines))
    print('=====RES_END=====')


In [ ]:
run_kaggle(['GPR'], ('GPR',), DATA, 'YS')